In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.constants import hbar, k as kB
import json


# Physical constants
M_K39 = 39 * 1.6605e-27            # Mass of K39 in kg
A0 = 5.29177e-11                  # Bohr radius


# Dimensionless units
length_unit = 1e-6                                           # m
energy_unit = hbar**2 / (M_K39 * length_unit**2)             # J
time_unit   = hbar / energy_unit                             # s
temp_unit   = energy_unit / kB                               # K



path=r"D:\Users\Public\Documents\Louis\Simulations data\Data - relaxation\T=8.04-mu_ini=0.32-n0=20.00-gamma=0.0300-dt=0.00163" # Initial state folder

In [ ]:

with open(path+r"\metadata_initial_state.json", 'r') as file:
    data = json.load(file)


L_phys= data["L_phys"]
T_phys=data["T_phys"]
n0_phys=data["n0_phys"]
omega_z=data["omega_z"]
mu_ini=data["mu_ini"]
L=data["L"]
T=data["T"]
gamma_ini=data["gamma_ini"]
dt=data["dt"]
n_steps=data["n_steps"]
N_grid=data["N_grid"]
k_cut=data["k_cut"]
save_every=data["save_every"]
dx=L/N_grid

psi_storage = np.memmap(path+r'/psi_gound_state.dat',
                        dtype=np.complex128,
                        mode='r',
                        shape=(n_steps//save_every+1,N_grid,N_grid))

Psi_s=[]
for i in range(n_steps//save_every):
    psi_t= psi_storage[i]
    Psi_s.append(psi_t)

In [ ]:
import numpy as np
from numpy.fft import fft2, fftshift, fftfreq


def get_radially_averaged_sk(psi, dx, L, n_bins=100, k_max=None):

    psi = np.asarray(psi)

    N_grid = psi.shape[0]
    density = np.abs(psi)**2
    n_avg = density.mean()

    if n_avg <= 0:
        raise ValueError("Average density is zero.")

    delta_n = density - n_avg

    Sk_2d = (
        fftshift(np.abs(fft2(delta_n))**2)
        * dx**2
        / (n_avg * N_grid**2)
    )

    k_1d = fftshift(fftfreq(N_grid, d=dx) * 2*np.pi)
    kx, ky = np.meshgrid(k_1d, k_1d, indexing="ij")
    k_map = np.sqrt(kx**2 + ky**2)

    k_min = 2*np.pi / L
    k_max = np.max(k_1d) if k_max is None else k_max

    bins = np.linspace(k_min, k_max, n_bins + 1)
    centers = 0.5 * (bins[:-1] + bins[1:])

    idx = np.digitize(k_map.ravel(), bins)

    sk_sum = np.bincount(idx, weights=Sk_2d.ravel(), minlength=n_bins + 2)
    counts = np.bincount(idx, minlength=n_bins + 2)

    sk = np.full(n_bins, np.nan)
    valid = counts[1:n_bins+1] > 0
    sk[valid] = sk_sum[1:n_bins+1][valid] / counts[1:n_bins+1][valid]

    finite = np.isfinite(sk)
    return centers[finite], sk[finite]


def get_averaged_sk(psi_list, dx, L, n_bins=100, k_max=None):

    sk_list = []
    k_ref = None

    for psi in psi_list:
        k, sk = get_radially_averaged_sk(
            psi, dx, L, n_bins=n_bins, k_max=k_max
        )

        if k_ref is None:
            k_ref = k

        sk_list.append(sk)

    sk_array = np.asarray(sk_list)

    mean = np.nanmean(sk_array, axis=0)
    err = np.nanstd(sk_array, axis=0) / np.sqrt(np.sum(np.isfinite(sk_array), axis=0))

    return k_ref, mean, err

The following lines load the $S_k(t)$ for each run

In [ ]:
import tqdm
from pathlib import Path

path_quench=r"YourQuenchPath"

with open(path_quench+r"/metadata_final_state.json", 'r') as file:
    data_quench = json.load(file)

mu_final=data_quench["mu_final"]
gamma_final=data_quench["gamma_final"]
n_steps_final=data_quench["n_steps_final"]
quench_step=data_quench["quench_step"]


n_runs=75 # Ajust the number of run you want to load
n_bins= N_grid // 2 - 1
print(n_bins)
sk=[]
print(n_steps_final//save_every+1)
for r in tqdm.tqdm(range(n_runs)):
    if r==8:
         continue
    sk_run=[]
    psi_storage = np.memmap(path_quench+f'/psi_final_run{r}.dat',
                        dtype=np.complex128,
                        mode='r',
                        shape=(n_steps_final//save_every+1,N_grid,N_grid))
    for i in range(n_steps_final//save_every-1):

        k_c, sk_r, sk_err=get_averaged_sk(psi_storage[i:i+1], dx, L, n_bins=n_bins) # radial average
        sk_run.append(sk_r)

    sk.append(sk_run)
    del psi_storage


k_c, sk_mean_ini, sk_err = get_averaged_sk(
    Psi_s[-800:], dx, L, n_bins=n_bins)



save_path = Path(path_quench) / r"Pictures/"  # Path of the folder to save figures
save_path.mkdir(exist_ok=True)


sk_std=np.std(sk,axis=0)
sk_mean=np.mean(sk,axis=0)


with open(path_quench+r"\metadata_final_state.json", 'r') as file:
    data = json.load(file)


L_phys= data["L_phys"]
T_phys=data["T_phys"]
n0_phys=data["n0_phys"]
omega_z=data["omega_z"]
mu_ini=data["mu_ini"]
mu_final=data["mu_final"]
L=data["L"]
T=data["T"]
gamma_ini=data["gamma_ini"]
gamma_final=data["gamma_final"]
dt=data["dt"]
n_steps_final=data["n_steps_final"]
quench_step=data["quench_step"]
N_grid=data["N_grid"]
k_cut=data["k_cut"]
save_every=data["save_every"]
dx=L/N_grid

osc_len_z  = np.sqrt(hbar / (M_K39 * omega_z))

g_ini=mu_ini/n0_phys*1e12
g_phys_ini    = g_ini * (energy_unit * length_unit**2)
a_ini=g_phys_ini*osc_len_z/((hbar**2 / M_K39) * np.sqrt(8 * np.pi))

g_final=mu_final/n0_phys*1e12
g_phys_final = g_final * (energy_unit * length_unit**2)
a_final=g_phys_final*osc_len_z/((hbar**2 / M_K39) * np.sqrt(8 * np.pi))

delta_a=(a_final/a_ini)


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl
import matplotlib.cm as cm
from matplotlib.ticker import AutoMinorLocator
mpl.rcParams.update(mpl.rcParamsDefault)
mpl.rcParams.update({
    "font.family": "serif",
    "mathtext.fontset": "cm",
    "font.size": 13,
    "axes.labelsize": 15,
    "axes.linewidth": 1.1,
    "xtick.direction": "in",
    "ytick.direction": "in",
    "xtick.top": True,
    "ytick.right": True,
    "xtick.minor.visible": True,
    "ytick.minor.visible": True,
    "xtick.major.size": 5,
    "ytick.major.size": 5,
    "xtick.minor.size": 3,
    "ytick.minor.size": 3,
    "legend.frameon": False,
    "lines.linewidth": 1.8,
})


n_bins= N_grid // 2 - 1

k_1d = np.fft.fftshift(
    np.fft.fftfreq(N_grid, d=dx) * 2 * np.pi
)

k_map = np.sqrt(np.add.outer(k_1d**2, k_1d**2))

bins  = np.linspace(2 * np.pi / L, np.max(k_1d), n_bins + 1)
k_c   = (bins[:-1] + bins[1:]) / 2

bin_indices = np.digitize(k_map.ravel(), bins).clip(1, n_bins)
counts = np.bincount(bin_indices, minlength=n_bins + 1)

mask = counts[1:n_bins + 1] > 0
k_c  = k_c[mask]

pre_quench_mean,  pre_quench_std  = [], []
post_quench_mean, post_quench_std = [], []

for i in range(sk_mean.shape[0]):
    if i < quench_step // save_every:
        pre_quench_mean.append(sk_mean[i][mask])
        pre_quench_std.append(sk_std[i][mask])
    else:
        post_quench_mean.append(sk_mean[i][mask])
        post_quench_std.append(sk_std[i][mask])


t_scale = dt * save_every * time_unit * 1e3   # ms per saved step

t_start_pre = 0.0
t_end_post  = max(
    (len(pre_quench_mean) + len(post_quench_mean) - 1) * t_scale,
    t_scale
)

t_quench_ms = quench_step * dt * time_unit * 1e3

norm_global  = mpl.colors.Normalize(vmin=t_start_pre, vmax=t_end_post)
quench_frac  = np.clip(
    (t_quench_ms - t_start_pre) / (t_end_post - t_start_pre), 0.0, 1.0
)

colors_list = [
    (0.0,         cm.Blues(0.35)),
    (quench_frac, cm.Blues(0.95)),
    (quench_frac, cm.Reds(0.35)),
    (1.0,         cm.Reds(0.95)),
]
custom_cmap = mpl.colors.LinearSegmentedColormap.from_list("quench_map", colors_list)
sm_global   = cm.ScalarMappable(norm=norm_global, cmap=custom_cmap)

plot_every = max(1, len(pre_quench_mean)  // 50)   
plot_every_post = max(1, len(post_quench_mean) // 300) 

pre_plot_idx  = list(range(0, len(pre_quench_mean),  plot_every))
post_plot_idx = list(range(0, len(post_quench_mean), plot_every_post))

print(f"Plotting {len(pre_plot_idx)} pre + {len(post_plot_idx)} post curves")


fig, ax = plt.subplots(figsize=(5.8, 4.0))

for idx in pre_plot_idx:
    t = idx * t_scale
    ax.plot(k_c, pre_quench_mean[idx], color=sm_global.to_rgba(t), alpha=0.9)

for idx in post_plot_idx:
    t = (idx + len(pre_quench_mean)) * t_scale
    ax.plot(k_c, post_quench_mean[idx], color=sm_global.to_rgba(t), alpha=0.9)


ax.set_xlim(k_c[0], k_cut)
ax.set_ylim(np.nanmin(sk_mean) * 0.8, np.nanmax(sk_mean) * 1.2)

ax.set_xlabel(r"$k$")
ax.set_ylabel(r"$\langle S(k) \rangle_{\mathrm{ens}}$")

ax.xaxis.set_minor_locator(AutoMinorLocator())
ax.yaxis.set_minor_locator(AutoMinorLocator())

ax.text(
    0.03, 0.95,
    rf"$N_{{\rm run}}={n_runs}$",
    transform=ax.transAxes,
    ha="left", va="top",
)


cbar = fig.colorbar(sm_global, ax=ax, orientation="vertical", pad=0.03, shrink=0.85)
cbar.set_label(r"Time $t$ (ms)")

cbar.ax.axhline(t_quench_ms, color="black", linewidth=1.5, linestyle="--")

cbar.ax.annotate(
    r"$t_{\mathrm{quench}}$",
    xy=(1.0, t_quench_ms),
    xycoords=("axes fraction", "data"),
    xytext=(6, 0),
    textcoords="offset points",
    va="center", ha="left",
    color="black",
    annotation_clip=False,
)


fig.tight_layout()
# plt.savefig(save_path / r"Sk_dynamics.pdf", bbox_inches="tight")
# plt.savefig(save_path / r"Sk_dynamics.png", bbox_inches="tight")
plt.show()

In [ ]:
from scipy.ndimage import gaussian_filter1d
from scipy.optimize import curve_fit
import math
import matplotlib as mpl
from matplotlib.ticker import AutoMinorLocator

mpl.rcParams.update({
    "font.family": "serif",
    "mathtext.fontset": "cm",
    "font.size": 13,
    "axes.labelsize": 15,
    "axes.linewidth": 1.1,
    "xtick.direction": "in",
    "ytick.direction": "in",
    "xtick.top": True,
    "ytick.right": True,
    "xtick.minor.visible": True,
    "ytick.minor.visible": True,
    "xtick.major.size": 5,
    "ytick.major.size": 5,
    "xtick.minor.size": 3,
    "ytick.minor.size": 3,
    "legend.frameon": False,
})


num_time_steps = sk_mean.shape[0]
times_ms    = np.arange(num_time_steps) * dt * save_every * time_unit * 1e3
t_quench_ms = quench_step * dt * time_unit * 1e3
post_idx    = np.searchsorted(times_ms, t_quench_ms)
window_end_idx = -1


dk               = 2*np.pi / L
valid_mask       = (k_c > dk) & (k_c < k_cut)

available_k_idx  = np.where(valid_mask)[0]

mask= [np.abs(np.mean(sk_mean[:, k_idx])) > 0 for k_idx in available_k_idx]
good_k_idx = available_k_idx[mask]

n_panels = len(good_k_idx)


n_panels = len(available_k_idx)
ncols    = 5                              
nrows    = math.ceil(n_panels / ncols)

fig, axes = plt.subplots(
    nrows, ncols,
    figsize=(6.75 * ncols / 2, 2.4 * nrows),
    sharey=False
)
axes_flat = np.array(axes).flatten()      

legend_placed = False

for j, k_idx in enumerate(good_k_idx):
    ax    = axes_flat[j]
    k_val = k_c[k_idx]
    s_t     = sk_mean[:, k_idx]
    s_t_std = sk_std[:, k_idx]

    post_times_shifted = times_ms[post_idx:window_end_idx] - t_quench_ms
    post_times_abs     = times_ms[post_idx:window_end_idx]

    ax.plot(times_ms, s_t, color='tab:blue', lw=1.5, label=r'$S(k,t)$')
    ax.fill_between(
        times_ms,
        s_t - s_t_std,
        s_t + s_t_std,
        color='tab:blue', alpha=0.2, edgecolor=None
    )

    ax.axvline(t_quench_ms, color="#d62728", lw=0.8, ls="--", zorder=50)

    ax.xaxis.set_minor_locator(AutoMinorLocator())
    ax.yaxis.set_minor_locator(AutoMinorLocator())
    # ax.set_xlim(t_quench_ms - 4.0, t_quench_ms + 20)
    ax.set_title(
    rf"$k={k_val:.2f}$")
    if j >= (nrows - 1) * ncols:
        ax.set_xlabel(r"Time $t$ (ms)")

    if j % ncols == 0:
        ax.set_ylabel(r"$\langle S(k,t)\rangle_{\rm ens}$")
    else:
        ax.tick_params(labelleft=False)

    if not legend_placed:
        handles = [
            mpl.lines.Line2D([0], [0], color="tab:blue",   lw=1.5,          label=r"$S(k,t)$"),
            mpl.lines.Line2D([0], [0], color="#d62728",    lw=0.8, ls="--", label=r"$t_{\rm quench}$"),
        ]
        ax.legend(handles=handles, loc="lower right", handlelength=1.6,
                  borderpad=0.4, labelspacing=0.3)
        legend_placed = True

for j in range(n_panels, len(axes_flat)):
    axes_flat[j].set_visible(False)


fig.tight_layout(pad=0.8, w_pad=0.6)
# plt.savefig(full_path / "Sk_relaxation_panels_NoFit.pdf", bbox_inches="tight")
# plt.savefig(full_path / "Sk_relaxation_panels_NoFit.png", dpi=300, bbox_inches="tight")
plt.show()



# Oscillations extraction

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

import math
import matplotlib as mpl

from scipy.optimize import curve_fit
from matplotlib.ticker import AutoMinorLocator


mpl.rcParams.update({
    "font.family": "serif",
    "mathtext.fontset": "cm",
    "font.size": 13,
    "axes.labelsize": 15,
    "axes.linewidth": 1.1,
    "xtick.direction": "in",
    "ytick.direction": "in",
    "xtick.top": True,
    "ytick.right": True,
    "xtick.minor.visible": True,
    "ytick.minor.visible": True,
    "xtick.major.size": 5,
    "ytick.major.size": 5,
    "xtick.minor.size": 3,
    "ytick.minor.size": 3,
    "legend.frameon": False,
})


def damped_osc_drift(t, A, tau, omega, phi, B, C):

    return A * np.exp(-t / tau) * np.cos(omega * t + phi) + (B * t + C)


num_time_steps = sk_mean.shape[0]

times_ms = (
    np.arange(num_time_steps)
    * dt
    * save_every
    * time_unit
    * 1e3
)

t_quench_ms = quench_step * dt * time_unit * 1e3

times_rel_ms = (times_ms - t_quench_ms) * 1e-3 / time_unit


dk = 2 * np.pi / L

valid_mask = (k_c > dk) & (k_c < k_cut)
available_k_idx = np.where(valid_mask)[0]


all_k_vals = []
all_taus = []
all_tau_errs = []
all_A = []
fit_results = {}

for k_idx in available_k_idx:
    k_val = k_c[k_idx]
    y = sk_mean[:, k_idx]
    yerr = sk_std[:, k_idx]
    t = times_rel_ms 

    valid = np.isfinite(y) & np.isfinite(yerr) & (yerr > 0) & (t > 0)

    t_fit = t[valid]
    y_fit = y[valid]
    yerr_fit = yerr[valid]

    if len(t_fit) < 15:
        continue

    E_guess = 2.0 * np.sqrt((k_val**2 / 2.0) * (k_val**2 / 2.0 + 2.0 * mu_ini))
    A_guess = 0.5 * (np.max(y_fit) - np.min(y_fit))
    tau_guess = 0.5 * (t_fit[-1] - t_fit[0])
    phi_guess = 0.0
    B_guess = 0.0
    C_guess = np.mean(y_fit[-max(5, len(y_fit)//5):])

    p0 = [A_guess, tau_guess, E_guess, phi_guess, B_guess, C_guess]
    bounds = (
        [-np.inf, 1e-4, 0.0, -np.pi, -np.inf, -np.inf],
        [ np.inf, np.inf, np.inf,  np.pi,  np.inf,  np.inf]
    )

    try:
        popt, pcov = curve_fit(
            damped_osc_drift,
            t_fit,
            y_fit,
            p0=p0,
            sigma=yerr_fit,
            absolute_sigma=True,
            bounds=bounds,
            maxfev=30000
        )

        A_fit, tau_fit, omega_fit, phi_fit, B_fit, C_fit = popt
        perr = np.sqrt(np.diag(pcov))
        tau_err = perr[1]
        omega_err = perr[2]  

        if tau_fit <= 0:
            continue

        all_k_vals.append(k_val)
        all_taus.append(tau_fit)
        all_tau_errs.append(tau_err)
        all_A.append(A_fit)

        y_pred = damped_osc_drift(t_fit, *popt)
        residuals = (y_fit - y_pred) / yerr_fit
        chi2_red = np.sum(residuals**2) / (len(t_fit) - len(p0))

        ss_res = np.sum((y_fit - y_pred)**2)
        ss_tot = np.sum((y_fit - np.mean(y_fit))**2)
        r_squared = 1.0 - (ss_res / ss_tot) if ss_tot > 0 else 0.0

        fit_results[k_idx] = {
            "A": A_fit,
            "tau": tau_fit,
            "omega": omega_fit,
            "phi": phi_fit,
            "B": B_fit,
            "C": C_fit,
            "tau_err": tau_err,
            "omega_err": omega_err,
            "chi2_red": chi2_red,
            "r_squared": r_squared
        }

    except RuntimeError:
        continue

all_k_vals = np.array(all_k_vals)
all_taus = np.array(all_taus)
all_tau_errs = np.array(all_tau_errs)


mask = [np.mean(sk_mean[:, k_idx]) > 0 for k_idx in available_k_idx]
good_k_idx = available_k_idx[mask]

n_panels = len(good_k_idx)
ncols = 5
nrows = math.ceil(n_panels / ncols)


fig, axes = plt.subplots(
    nrows, ncols,
    figsize=(6.75 * ncols / 2, 2.4 * nrows),
    sharey=False
)

axes_flat = np.array(axes).flatten()

legend_placed = False

all_k = []
all_omega = []
all_omega_err = []
all_tau = []
all_tau_err = []
all_r_squared = []
all_chi2_red = []

for j, k_idx in enumerate(good_k_idx):
    ax = axes_flat[j]
    k_val = k_c[k_idx]

    raw_s = sk_mean[:, k_idx]

    n_tail = max(10, len(raw_s) // 5)
    norm_factor = np.mean(raw_s[-n_tail:])

    s_t = raw_s / norm_factor
    s_t_std = sk_std[:, k_idx] / norm_factor

    ax.plot(times_rel_ms, s_t, color="tab:blue", lw=1.5)
    ax.fill_between(
        times_rel_ms,
        s_t - s_t_std,
        s_t + s_t_std,
        color="tab:blue",
        alpha=0.2
    )
    ax.axvline(0, color="#d62728", ls="--", lw=0.8)

    if k_idx in fit_results:
        p = fit_results[k_idx]

        t_fit_eval = np.linspace(0, times_rel_ms[-1], 1000)
        y_fit = damped_osc_drift(
            t_fit_eval, p["A"], p["tau"], p["omega"], p["phi"], p["B"], p["C"]
        )
        y_fit /= norm_factor
        ax.plot(t_fit_eval, y_fit, color="black", lw=1)

        all_k.append(k_val)
        all_omega.append(p["omega"])
        all_omega_err.append(p["omega_err"])
        all_tau.append(p["tau"])
        all_tau_err.append(p["tau_err"])
        all_r_squared.append(p["r_squared"])
        all_chi2_red.append(p["chi2_red"])

        # Mark the 1/e decay point
        ax.axvline(
            p["tau"],
            color="#2c3fa0",
            ls=":",
            lw=1
        )

        ax.set_title(
            rf"$k={k_val:.2f}$"
            "\n"
            rf"$\tau={p['tau']:.2f}\pm{p['tau_err']:.2f}$",
            pad=4
        )

    else:
        ax.set_title(rf"$k={k_val:.2f}$")

    ax.set_xlabel("Time")

    if j % 5 == 0:
        ax.set_ylabel(r"norm. $\langle S(k,t)\rangle_{\rm ens}$")
    else:
        ax.set_ylabel(r"")
    ax.set_xlim(-10,22)
    ax.xaxis.set_minor_locator(AutoMinorLocator())
    ax.yaxis.set_minor_locator(AutoMinorLocator())

    if not legend_placed:
        ax.legend([
            mpl.lines.Line2D([0], [0], color="tab:blue", lw=1.5),
            mpl.lines.Line2D([0], [0], color="black", lw=2),
            mpl.lines.Line2D([0], [0], color="#d62728", lw=0.8, ls="--"),
            mpl.lines.Line2D([0], [0], color="#2c3fa0", lw=0.8, ls=":")
        ], [
            "SPGPE",
            "damped osc. fit",
            "quench (t=0)",
            r"$1/e$ decay"
        ], loc="best")

        legend_placed = True

fig.tight_layout()
plt.show()

all_k1 = np.asarray(all_k)
all_omega1 = np.asarray(all_omega) / (time_unit * 1e3)
all_omega_err1 = np.asarray(all_omega_err) / (time_unit * 1e3)

all_tau1 = np.asarray(all_tau) * time_unit * 1e3
all_tau_err1 = np.asarray(all_tau_err) * time_unit * 1e3

all_r_squared1 = np.asarray(all_r_squared)


remove_indices_omega = []  # Indices to exclude from Frequency (top plot)
remove_indices_tau   = []  # Indices to exclude from Lifetime (bottom plot)

base_mask = all_r_squared1 > 0.0

# Create individual boolean masks
mask_omega = base_mask.copy()
mask_omega[remove_indices_omega] = False

mask_tau = base_mask.copy()
mask_tau[remove_indices_tau] = False

k_sak = np.linspace(all_k[0], all_k[-1], 300)
sakharov = 2 * np.sqrt((k_sak**2 / 2) * ((k_sak**2 / 2) + 2 * mu_final)) / (time_unit * 1e3)
sakharov2 = 2 * np.sqrt((k_sak**2 / 2) * ((k_sak**2 / 2) + 2 * mu_ini)) / (time_unit * 1e3)

plt.rcParams.update({
    "font.family": "serif",
    "font.serif": ["Times New Roman", "DejaVu Serif"],
    "mathtext.fontset": "stix",          
    "font.size": 10,                 
    "axes.labelsize": 10,
    "xtick.labelsize": 8,
    "ytick.labelsize": 8,
    "xtick.direction": "in",            
    "ytick.direction": "in",
    "xtick.top": True,                  
    "ytick.right": True,
    "xtick.major.size": 4,
    "xtick.minor.size": 2,
    "ytick.major.size": 4,
    "ytick.minor.size": 2,
    "axes.grid": False,                 
})

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(3.375, 4.5), sharex=True)

ax1.plot(k_sak, sakharov, color='black', linestyle='-', linewidth=1.2, label='Sakharov oscillation final')
ax1.plot(k_sak, sakharov2, color='black', linestyle='--', linewidth=1.2, label='Sakharov oscillation ini')

ax1.errorbar(
    all_k1[mask_omega], 
    all_omega1[mask_omega], 
    yerr=all_omega_err1[mask_omega],
    fmt='o', 
    mfc='none', 
    mec='crimson', 
    ecolor='crimson', 
    ms=4, 
    mew=0.8, 
    elinewidth=0.8, 
    capsize=2, 
    label='Data'
)

ax1.set_ylabel(r'Frequency $\omega$')
ax1.legend(loc='upper left', frameon=False)
ax1.set_ylim(0,20)

ax2.plot(k_sak, 5 * (sakharov**-1)*2, color='black', linestyle='--', linewidth=1.2, label=r'$\tau\propto \omega_{bog}^{-1}$')

ax2.errorbar(
    all_k1[mask_tau], 
    all_tau1[mask_tau], 
    yerr=all_tau_err1[mask_tau],
    fmt='s', 
    mfc='none', 
    mec='royalblue', 
    ecolor='royalblue', 
    ms=4, 
    mew=0.8, 
    elinewidth=0.8, 
    capsize=2, 
    label='Data'
)

ax2.set_ylim(0,15)
ax2.set_xlabel(r'Wavevector $k$')
ax2.set_ylabel(r'Lifetime $\tau$')
ax2.legend(loc='upper right', frameon=False)

plt.subplots_adjust(hspace=0.08) 
plt.show()

# Relaxation extraction tanh fit

In [ ]:
import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt

from scipy.optimize import curve_fit
from scipy.ndimage import gaussian_filter1d
from matplotlib.ticker import AutoMinorLocator

mpl.rcParams.update({
    "font.family": "serif",
    "mathtext.fontset": "cm",
    "font.size": 13,
    "axes.labelsize": 15,
    "axes.linewidth": 1.1,
    "xtick.direction": "in",
    "ytick.direction": "in",
    "xtick.top": True,
    "ytick.right": True,
    "xtick.minor.visible": True,
    "ytick.minor.visible": True,
    "xtick.major.size": 5,
    "ytick.major.size": 5,
    "xtick.minor.size": 3,
    "ytick.minor.size": 3,
    "legend.frameon": False,
})


def tanh_relax(t,t0, S0, dS, coef, tau):
    return S0 + 0.5 * dS * (1.0 + np.tanh((t - (t0+coef*tau)) / tau))

num_time_steps = sk_mean.shape[0]

times_ms = (
    np.arange(num_time_steps)
    * dt
    * save_every
    * time_unit
    * 1e3
)

t_quench_ms = quench_step * dt * time_unit * 1e3

# ----------------------------------------------------------
# k-range
# ----------------------------------------------------------
dk = np.pi / L

valid_mask = (k_c > dk) & (k_c < k_cut)
available_k_idx = np.where(valid_mask)[0]

# ----------------------------------------------------------
# Fit tau(k)
# ----------------------------------------------------------
all_k_vals = []
all_taus = []
all_tau_errs = []

fit_results = {}

for k_idx in available_k_idx:

    y = sk_mean[:, k_idx]
    yerr = sk_std[:, k_idx]
    t = times_ms

    valid = np.isfinite(y) & np.isfinite(yerr) & (yerr > 0)

    y = y[valid][:-70]
    yerr = yerr[valid][:-70]
    yerr/=np.sqrt(n_runs)
    t = t[valid][:-70]

    if len(t) < 30:
        continue


    pre_mean = np.mean(y[:max(10, len(y)//5)])
    post_mean = np.mean(y[-max(10, len(y)//5):])

    S0_guess = pre_mean
    dS_guess = post_mean - pre_mean
    tc_guess = t_quench_ms
    coef=1
    tau_guess = 0.05 * (t[-1] - t[0])

    p0 = [S0_guess, dS_guess, coef, tau_guess]

    bounds = (
        [-np.inf, -np.inf, coef*0.9, 1e-6],
        [ np.inf,  np.inf, coef*1.5, np.inf]
    )

    try:

        popt, pcov = curve_fit(
            lambda t, S0, dS, coef, tau: tanh_relax(t,t_quench_ms,S0, dS, coef, tau),
            t,
            y,
            p0=p0,
            sigma=yerr,
            absolute_sigma=True,
            bounds=bounds,
            maxfev=50000
        )

        S0_fit, dS_fit, tc_fit, tau_fit = popt
        perr = np.sqrt(np.diag(pcov))

        tau_err = perr[3]

        if tau_fit <= 0:
            continue

        all_k_vals.append(k_c[k_idx])
        all_taus.append(tau_fit)
        all_tau_errs.append(tau_err)

        fit_results[k_idx] = {
            "S0": S0_fit,
            "dS": dS_fit,
            "tc": tc_fit,
            "tau": tau_fit,
            "tau_err": tau_err,
        }

    except RuntimeError:
        continue

all_k_vals = np.array(all_k_vals)
all_taus = np.array(all_taus)
all_tau_errs = np.array(all_tau_errs)

# ----------------------------------------------------------
# Choose example panels
# ----------------------------------------------------------
# num_panels = min(4, len(available_k_idx))

# chosen_idx = np.sort(
#     available_k_idx[
#         np.random.randint(0, len(available_k_idx), num_panels)
#     ]
# )
mask= [np.mean(sk_mean[:, k_idx]) > 0 for k_idx in available_k_idx]
good_k_idx = available_k_idx[mask]

n_panels = len(good_k_idx)
ncols    = 5                              # panels per row — adjust as needed
nrows    = math.ceil(n_panels / ncols)

# ----------------------------------------------------------
# Plot time traces + fits
# ----------------------------------------------------------
fig, axes = plt.subplots(
    nrows, ncols,
    figsize=(6.75 * ncols / 2, 2.4 * nrows),
    sharey=False
)

axes_flat = np.array(axes).flatten()      # always 1-D for easy indexing

legend_placed = False


for j, k_idx in enumerate(good_k_idx):
    ax    = axes_flat[j]
    k_val = k_c[k_idx]

    raw_s = sk_mean[:, k_idx]
    n_tail = max(10, len(raw_s) // 5)
    norm_factor = np.mean(raw_s[-n_tail:])

    s_t = raw_s/norm_factor#gaussian_filter1d(raw_s, sigma=5) / norm_factor
    s_t_std = sk_std[:, k_idx] / norm_factor  
    s_t_sem=s_t_std/np.sqrt(n_runs)

    ax.plot(times_ms, s_t, color="tab:blue", lw=1.5)
    ax.fill_between(
        times_ms,
        s_t - s_t_std,
        s_t + s_t_std,
        color="tab:blue",
        alpha=0.2
    )
    ax.axvline(t_quench_ms, color="#d62728", ls="--", lw=0.8)


    if k_idx in fit_results:
        p = fit_results[k_idx]
        t_fit = np.linspace(times_ms[0], times_ms[-1], 1000)
        y_fit = tanh_relax(t_fit, t_quench_ms,p["S0"], p["dS"], p["tc"], p["tau"])
        y_fit /= norm_factor  
        ax.plot(t_fit, y_fit, color="black", lw=1)

        # ax.axvline(
        #     p["tc"],
        #     color="k",
        #     ls="--",
        #     lw=1
        # )
        ax.axvline(
            p["tc"]*p["tau"]+t_quench_ms + 0.773 * p["tau"],
            color="#2c3fa0",
            ls=":",
            lw=1
        )

        ax.set_title(
            rf"$k={k_val:.2f}$"
            "\n"
            rf"$\tau={p['tau']:.2f}\pm{p['tau_err']:.2f}\,\mathrm{{ms}}$",
            pad=4
        )

    else:
        ax.set_title(rf"$k={k_val:.2f}$")

    ax.set_xlabel("Time (ms)")
    # ax.set_ylim(0,1)
    ax.set_xlim(t_quench_ms - 2.0, t_quench_ms + 6)

    if j %5== 0:
        ax.set_ylabel(r"norm. $\langle S(k,t)\rangle_{\rm ens}$")
    else:
        ax.set_ylabel(r"")
        #ax.tick_params(labelleft=False)

    ax.xaxis.set_minor_locator(AutoMinorLocator())
    ax.yaxis.set_minor_locator(AutoMinorLocator())

    if not legend_placed:
        ax.legend([
            mpl.lines.Line2D([0],[0],color="tab:blue",lw=1.5),
            mpl.lines.Line2D([0],[0],color="black",lw=2),
            mpl.lines.Line2D([0],[0],color="#d62728",lw=0.8,ls="--"),
            mpl.lines.Line2D([0],[0],color="#2c3fa0",lw=0.8,ls=":")
        ], [
            "SPGPE",
            "tanh fit",
            "quench",
            r"$1/e$ point"
        ], loc="best")

        legend_placed = True

fig.tight_layout()
# plt.savefig(save_path / r"k_relaxation_panels_fit.pdf", bbox_inches="tight")
# plt.savefig(save_path / r"Sk_relaxation_panels_fit.png", dpi=300, bbox_inches="tight")

plt.show()
